# step04 - SUNA / CTD time-lag and alignment diagnostic

Answers the question raised on the SUNA: is the nitrate signal shifted in time relative to the CTD depth, and if so, is it a clock-synchronisation problem or the SUNA's own response time?

For each cast this notebook:

1. plots raw nitrate and depth against time on a dual axis (the figure Pierre asked for);
2. computes the time lag two ways: the direct peak-to-peak offset (time of max nitrate minus time of the deepest point) and a cross-correlation lag (more robust);
3. overlays the downcast and upcast nitrate-versus-depth profiles. A pure clock offset shifts both legs the same way, so a single time shift collapses them; a response-time lag makes the legs diverge (hysteresis), so a single clock offset will not fully align them;
4. reports a first-guess `SUNA_CLOCK_OFFSET_S` per cast and a cruise median.

Sign convention used throughout: a **positive lag means the nitrate occurs after the depth**, i.e. the SUNA nitrate lags the CTD.

Input: the co-located SUNA L2 product (`L2/SUNA/<CRUISE>__suna_L2_colocated.csv`), which carries `time_utc`, `nitrate_uM`, `pressure_db`, the `L1_clean` flag, `cast`, and `leg`. This notebook is additive: it reads that product and writes only figures and a summary CSV under `_audit/suna/time_lag/`. It changes no existing output.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve the repo root so `import project_paths` works from the notebook folder.
_here = Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / 'project_paths.py').exists():
        sys.path.insert(0, str(_cand))
        break
import project_paths as pp

# ---- settings ----
CRUISE_ID = 'P45_06'      # the only per-cruise switch
DT_S       = 1.0          # uniform grid step for cross-correlation (seconds)
MAX_LAG_S  = 120.0        # search window for the lag (seconds)
LEG_AGREE_TOL_S = 15.0    # |down lag - up lag| below this reads as a consistent (clock-like) offset

cruise = pp.cruise_dir(CRUISE_ID)
colocated = cruise / 'L2' / 'SUNA' / f'{CRUISE_ID}__suna_L2_colocated.csv'
out_dir = cruise / '_audit' / 'suna' / 'time_lag'
out_dir.mkdir(parents=True, exist_ok=True)
print('reading:', colocated)
print('writing:', out_dir)

In [ ]:
def pick(df, *cands):
    """Return the first column present, by exact then case-insensitive match. Mirrors suncat.util.pick,
    inlined so this notebook runs even before `pip install -e .`."""
    lower = {c.lower(): c for c in df.columns}
    for c in cands:
        if c in df.columns:
            return c
        if c.lower() in lower:
            return lower[c.lower()]
    raise KeyError(f'none of {cands} in {list(df.columns)}')

raw = pd.read_csv(colocated)

# Resolve columns by candidate names, so the notebook survives small naming changes across cruises.
c_time = pick(raw, 'time_utc', 'utc_time', 'time')
c_no3  = pick(raw, 'nitrate_uM', 'no3_uM', 'nitrate', 'suna_nitrate')
c_pres = pick(raw, 'pressure_db', 'pressure_dbar', 'pressure', 'pres')
c_cast = pick(raw, 'cast', 'cast_id')
c_leg  = pick(raw, 'leg')
c_flag = None
for cand in ('L1_clean', 'suna_L1_clean', 'clean'):
    if cand in raw.columns:
        c_flag = cand
        break

df = pd.DataFrame({
    'time': pd.to_datetime(raw[c_time], errors='coerce'),
    'no3':  pd.to_numeric(raw[c_no3], errors='coerce'),
    'pres': pd.to_numeric(raw[c_pres], errors='coerce'),
    'cast': raw[c_cast].astype(str),
    'leg':  raw[c_leg].astype(str).str.lower(),
})
df['clean'] = raw[c_flag].astype(bool) if c_flag else True
df = df.dropna(subset=['time', 'no3', 'pres']).sort_values(['cast', 'time']).reset_index(drop=True)
casts = sorted(df['cast'].unique())
print(f'{len(df)} rows, {len(casts)} casts:', ', '.join(casts))

In [ ]:
import numpy as np
import pandas as pd


def _grid(t_sec, y, dt):
    """Interpolate y(t_sec) onto a uniform grid at step dt. Returns (grid, y_on_grid)."""
    t_sec = np.asarray(t_sec, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(t_sec) & np.isfinite(y)
    t_sec, y = t_sec[m], y[m]
    if len(t_sec) < 5 or t_sec.max() <= t_sec.min():
        return None, None
    order = np.argsort(t_sec)
    t_sec, y = t_sec[order], y[order]
    g = np.arange(t_sec.min(), t_sec.max(), dt)
    return g, np.interp(g, t_sec, y)


def _z(a):
    a = np.asarray(a, dtype=float)
    s = a.std()
    return (a - a.mean()) / (s if s else 1.0)


def xcorr_lag(t_sec, no3, pres, dt=1.0, max_lag_s=120.0):
    """Lag (seconds) that best aligns nitrate to pressure by cross-correlation.

    Positive lag means the nitrate signal occurs LATER than the depth signal, i.e. the SUNA
    nitrate LAGS the CTD depth. Returns (lag_seconds, correlation_at_best_lag).
    """
    g, n = _grid(t_sec, no3, dt)
    _, p = _grid(t_sec, pres, dt)
    if g is None:
        return np.nan, np.nan
    n, p = _z(n), _z(p)
    max_k = int(max_lag_s / dt)
    best_k, best_c = 0, -np.inf
    for k in range(-max_k, max_k + 1):
        if k >= 0:
            a, b = n[k:], p[:len(p) - k] if k > 0 else p
        else:
            a, b = n[:len(n) + k], p[-k:]
        L = min(len(a), len(b))
        if L < 10:
            continue
        c = np.corrcoef(a[:L], b[:L])[0, 1]
        if np.isfinite(c) and c > best_c:
            best_c, best_k = c, k
    return best_k * dt, best_c


def peak_lag(t_sec, no3, pres):
    """Pierre's direct metric: time of max nitrate minus time of deepest point (seconds).

    Positive means the nitrate peak occurs after the depth peak.
    """
    t_sec = np.asarray(t_sec, dtype=float)
    no3 = np.asarray(no3, dtype=float)
    pres = np.asarray(pres, dtype=float)
    if np.isfinite(pres).sum() < 5 or np.isfinite(no3).sum() < 5:
        return np.nan
    t_depth = t_sec[np.nanargmax(pres)]
    t_no3 = t_sec[np.nanargmax(no3)]
    return float(t_no3 - t_depth)

In [ ]:
# Build-time sign check, kept in the notebook so the convention is self-evident and self-verifying.
_t = np.arange(0, 300.0, 1.0)
_p = np.exp(-((_t - 100.0) ** 2) / (2 * 20.0 ** 2))
_n = np.exp(-((_t - 110.0) ** 2) / (2 * 20.0 ** 2))   # nitrate delayed by 10 s
_lag, _c = xcorr_lag(_t, _n, _p, dt=1.0, max_lag_s=60.0)
assert abs(_lag - 10.0) <= 1.0 and abs(peak_lag(_t, _n, _p) - 10.0) <= 1.0
print(f'sign check OK: a nitrate signal delayed 10 s is reported as +{_lag:.0f} s (lags depth)')

In [ ]:
def cast_frame(cast_id):
    d = df[df['cast'] == cast_id].copy()
    t0 = d['time'].iloc[0]
    d['t_sec'] = (d['time'] - t0).dt.total_seconds()
    return d

def analyse_cast(cast_id):
    d = cast_frame(cast_id)
    # Use only spectrally clean frames for the lag, so bad spectra do not pull the estimate.
    dc = d[d['clean']] if d['clean'].any() else d
    full_lag, full_c = xcorr_lag(dc['t_sec'], dc['no3'], dc['pres'], DT_S, MAX_LAG_S)
    pk = peak_lag(dc['t_sec'].to_numpy(), dc['no3'].to_numpy(), dc['pres'].to_numpy())
    down = dc[dc['leg'] == 'down']
    up = dc[dc['leg'] == 'up']
    d_lag = xcorr_lag(down['t_sec'], down['no3'], down['pres'], DT_S, MAX_LAG_S)[0] if len(down) > 10 else np.nan
    u_lag = xcorr_lag(up['t_sec'], up['no3'], up['pres'], DT_S, MAX_LAG_S)[0] if len(up) > 10 else np.nan
    if np.isfinite(d_lag) and np.isfinite(u_lag):
        verdict = ('consistent offset (clock-like)' if abs(d_lag - u_lag) <= LEG_AGREE_TOL_S
                   else 'leg-dependent (response-time / plumbing)')
    else:
        verdict = 'one leg only (cannot compare)'
    return {
        'cast': cast_id,
        'n_clean': int(dc['clean'].sum()) if dc['clean'].dtype == bool else len(dc),
        'peak_lag_s': round(pk, 1) if np.isfinite(pk) else np.nan,
        'xcorr_lag_s': round(full_lag, 1) if np.isfinite(full_lag) else np.nan,
        'xcorr_r': round(full_c, 3) if np.isfinite(full_c) else np.nan,
        'down_lag_s': round(d_lag, 1) if np.isfinite(d_lag) else np.nan,
        'up_lag_s': round(u_lag, 1) if np.isfinite(u_lag) else np.nan,
        'verdict': verdict,
        # First guess for step02: shift SUNA time to remove the lag. Verify the sign against step02.
        'recommended_offset_s': round(-full_lag, 1) if np.isfinite(full_lag) else np.nan,
    }

summary = pd.DataFrame([analyse_cast(c) for c in casts])
summary

In [ ]:
def make_figure(cast_id, save=True):
    d = cast_frame(cast_id)
    row = summary[summary['cast'] == cast_id].iloc[0]
    fig, (ax1, ax3) = plt.subplots(1, 2, figsize=(13, 5))

    # Panel A: nitrate and depth vs time (Pierre's plot).
    ax1.plot(d['time'], d['no3'], color='tab:blue', lw=1.2, label='Raw nitrate')
    ax1.set_ylabel('Raw nitrate (umol L-1)', color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')
    ax1.set_xlabel('Time (UTC)')
    ax2 = ax1.twinx()
    ax2.plot(d['time'], d['pres'], color='tab:orange', lw=1.2, label='Depth')
    ax2.set_ylabel('Pressure / depth (dbar)', color='tab:orange')
    ax2.tick_params(axis='y', labelcolor='tab:orange')
    ax2.invert_yaxis()  # deeper is down
    ax1.set_title(f'{cast_id}: nitrate and depth vs time')
    for lab in ax1.get_xticklabels():
        lab.set_rotation(30); lab.set_ha('right')

    # Panel B: nitrate vs depth, down and up overlaid (hysteresis = response-time lag).
    for leg, col in (('down', 'tab:blue'), ('up', 'tab:orange')):
        s = d[d['leg'] == leg]
        if len(s):
            ax3.plot(s['no3'], s['pres'], color=col, lw=1.1, label=leg)
    ax3.invert_yaxis()
    ax3.set_xlabel('Raw nitrate (umol L-1)')
    ax3.set_ylabel('Pressure / depth (dbar)')
    ax3.legend(title='leg')
    ax3.set_title('nitrate vs depth (down / up)')

    txt = (f"peak-to-peak lag: {row['peak_lag_s']} s\n"
           f"cross-corr lag: {row['xcorr_lag_s']} s (r={row['xcorr_r']})\n"
           f"down {row['down_lag_s']} s / up {row['up_lag_s']} s\n"
           f"{row['verdict']}")
    ax3.text(0.02, 0.02, txt, transform=ax3.transAxes, va='bottom', ha='left', fontsize=9,
             bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    fig.tight_layout()
    if save:
        fig.savefig(out_dir / f'{cast_id}__time_lag.png', dpi=150, bbox_inches='tight')
    return fig

# Preview the first cast.
_ = make_figure(casts[0]); plt.show()

In [ ]:
# Interactive per-cast window; falls back to saving every figure if ipywidgets is not available.
try:
    import ipywidgets as widgets
    from IPython.display import display
    dd = widgets.Dropdown(options=casts, description='cast')
    def _show(cast):
        make_figure(cast, save=False); plt.show()
    widgets.interact(_show, cast=dd)
except Exception as e:
    print('ipywidgets not available, saving all figures instead:', e)
    for c in casts:
        make_figure(c); plt.close('all')
    print('saved', len(casts), 'figures to', out_dir)

In [ ]:
# Save every figure and the summary CSV, and report the cruise-level recommendation.
for c in casts:
    make_figure(c); plt.close('all')
summary_path = out_dir / f'{CRUISE_ID}__suna_time_lag_summary.csv'
summary.to_csv(summary_path, index=False)

med_lag = summary['xcorr_lag_s'].median()
med_off = summary['recommended_offset_s'].median()
print('saved summary:', summary_path)
print(f'cruise median cross-corr lag: {med_lag:+.1f} s (nitrate lags depth when positive)')
print(f'cruise median recommended SUNA_CLOCK_OFFSET_S: {med_off:+.1f} s  (verify sign against step02)')
print('verdict counts:')
print(summary['verdict'].value_counts().to_string())

## How to read this

- **If the lag is roughly constant across casts and the down and up legs agree** (`consistent offset (clock-like)`), it points to a clock offset between the SUNA and the CTD. Set `SUNA_CLOCK_OFFSET_S` in step02 to the `recommended_offset_s` (verify the sign against how step02 applies it: after the correction the nitrate peak should sit on top of the depth peak), then re-run step02 and step03.
- **If the down and up legs disagree** (`leg-dependent`), a single clock shift will not fully align the profiles. That is the SUNA's response time (intake plumbing and internal averaging), and it also broadens the nitrate relative to the depth, visible as the wider blue trace in the time plot. That needs a response-time correction, not just a clock offset, so flag it rather than silently shifting the clock.

Honest limit: the SUNA frames are spaced on the order of ten seconds, so the lag is resolved only to roughly that interval. Treat a single-cast number as indicative and the cruise median as the figure to act on.

One piece is not in the data: the physical mount. The offset between the CTD intake and the SUNA optical path contributes to the apparent lag, so a photo of how the SUNA sits on the rosette (Pierre's third question) is the other half of the answer and should go alongside these numbers.